# Importing the necessary libraries

In [52]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
import optuna
import os

# Loading the datasets

In [53]:
blr_df = pd.read_csv('../Data/Processed/blr_df.csv')
hyd_df = pd.read_csv('../Data/Processed/hyd_df.csv')
pune_df = pd.read_csv('../Data/Processed/pune_df.csv')

In [54]:
blr_df

,Date,DPT,AP,WS,WSD,RH,LST
0,2003-01-01,12.498648,916.589625,0.608896,327.505071,93.194473,30.939528
1,2003-01-02,12.825784,918.004624,3.006925,290.310937,93.578218,31.179113
2,2003-01-03,12.970981,918.453619,2.872020,279.921010,94.227231,33.045705
3,2003-01-04,12.445614,917.966755,2.678422,274.952584,93.297114,38.266933
4,2003-01-05,14.071074,918.579933,3.104962,275.257170,94.353249,33.185394
...,...,...,...,...,...,...,...
6553,2020-12-25,13.500707,916.151180,2.152634,261.457477,95.470006,32.118758
6554,2020-12-26,12.901577,917.194971,2.312996,253.686322,95.361429,32.068009
6555,2020-12-27,12.950693,916.302854,2.260313,252.095788,94.854902,34.539327
6556,2020-12-28,10.830827,916.237697,2.003774,254.229795,93.468912,31.310399


In [55]:
pune_df

,Date,DPT,AP,WS,WSD,RH,LST
0,2003-01-01,7.100958,941.961323,0.608896,327.505071,91.114710,34.246204
1,2003-01-02,10.484796,942.675433,3.006925,290.310937,91.934631,35.734294
2,2003-01-03,12.606104,942.881457,2.872020,279.921010,92.393007,32.590960
3,2003-01-04,12.646417,943.009227,2.678422,274.952584,92.312594,35.435619
4,2003-01-05,13.066706,943.762527,3.104962,275.257170,92.569458,34.235347
...,...,...,...,...,...,...,...
6553,2020-12-25,12.096190,941.053885,2.152634,261.457477,92.849653,33.209944
6554,2020-12-26,13.381471,942.263093,2.312996,253.686322,93.400294,31.352555
6555,2020-12-27,13.819016,941.287692,2.260313,252.095788,93.560516,33.862652
6556,2020-12-28,13.831078,940.630939,2.003774,254.229795,93.895661,32.318171


In [56]:
hyd_df

,Date,DPT,AP,WS,WSD,RH,LST
0,2003-01-01,9.801921,952.321702,2.702138,228.989644,91.137446,35.253027
1,2003-01-02,12.894156,953.896739,2.974241,287.669929,92.938629,31.660535
2,2003-01-03,16.945643,954.471306,3.015529,308.536731,96.690377,30.085849
3,2003-01-04,17.761199,954.172469,1.865083,299.395258,98.016959,30.569864
4,2003-01-05,14.281283,954.395644,2.287668,265.010524,93.245268,31.878895
...,...,...,...,...,...,...,...
6553,2020-12-25,13.330685,951.670802,2.015309,292.172202,94.673562,30.396700
6554,2020-12-26,14.027632,952.521458,1.736367,284.297801,94.788311,32.262344
6555,2020-12-27,13.223768,951.342613,1.754041,288.307250,94.039085,33.796627
6556,2020-12-28,12.504071,951.101661,2.012608,299.893708,93.248170,31.748070


# Since we will be applying a RNN, we will need the Date column to be Datetime column

In [57]:
blr_df['Date'] = pd.to_datetime(blr_df['Date'])
blr_df = blr_df.sort_values('Date')

In [58]:
hyd_df['Date'] = pd.to_datetime(hyd_df['Date'])
hyd_df = hyd_df.sort_values('Date')

In [59]:
pune_df['Date'] = pd.to_datetime(pune_df['Date'])
pune_df = pune_df.sort_values('Date')

# Splitting the Input and Predictor Variables

In [60]:
features = ['AP', 'DPT', 'WS', 'WSD', 'RH']
target = 'LST'

In [61]:
blr_X = blr_df[features].values
blr_y = blr_df[target].values

In [62]:
hyd_X = hyd_df[features].values
hyd_y = hyd_df[target].values

In [63]:
pune_X = pune_df[features].values
pune_y = pune_df[target].values

# Scaling the data

In [64]:
scaler = StandardScaler()

In [65]:
blr_X_scaled = scaler.fit_transform(blr_X)
hyd_X_scaled = scaler.transform(hyd_X)
pune_X_scaled = scaler.transform(pune_X)

# Creating the sequences for RNN

In [66]:
def create_sequences(X, y, seq_length=30):
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_length):
        X_seq.append(X[i:i+seq_length])
        y_seq.append(y[i+seq_length])  # predict LST at t+1
    return np.array(X_seq), np.array(y_seq)

In [67]:
seq_len = 30

In [68]:
blr_X_seq, blr_y_seq = create_sequences(blr_X_scaled, blr_y, seq_len)
hyd_X_seq, hyd_y_seq = create_sequences(hyd_X_scaled, hyd_y, seq_len)
pune_X_seq, pune_y_seq = create_sequences(pune_X_scaled, pune_y, seq_len)

# Splitting in Training and Testing Data

In [69]:
split_idx = int(0.8 * len(blr_X_seq))
blr_X_train, blr_X_test = blr_X_seq[:split_idx], blr_X_seq[split_idx:]
blr_y_train, blr_y_test = blr_y_seq[:split_idx], blr_y_seq[split_idx:]

In [70]:
split_idx = int(0.8 * len(hyd_X_seq))
hyd_X_train, hyd_X_test = hyd_X_seq[:split_idx], hyd_X_seq[split_idx:]
hyd_y_train, hyd_y_test = hyd_y_seq[:split_idx], hyd_y_seq[split_idx:]

In [71]:
split_idx = int(0.8 * len(pune_X_seq))
pune_X_train, pune_X_test = pune_X_seq[:split_idx], pune_X_seq[split_idx:]
pune_y_train, pune_y_test = pune_y_seq[:split_idx], pune_y_seq[split_idx:]

# Converting it into Pytorch Tensor for Further Analysis

In [72]:
blr_X_train_tensor  = torch.tensor(blr_X_train, dtype=torch.float32)
blr_y_train_tensor  = torch.tensor(blr_y_train, dtype=torch.float32).unsqueeze(1)
blr_X_test_tensor   = torch.tensor(blr_X_test, dtype=torch.float32)
blr_y_test_tensor   = torch.tensor(blr_y_test, dtype=torch.float32).unsqueeze(1)

hyd_X_train_tensor  = torch.tensor(hyd_X_train, dtype=torch.float32)
hyd_y_train_tensor  = torch.tensor(hyd_y_train, dtype=torch.float32).unsqueeze(1)
hyd_X_test_tensor   = torch.tensor(hyd_X_test, dtype=torch.float32)
hyd_y_test_tensor   = torch.tensor(hyd_y_test, dtype=torch.float32).unsqueeze(1)

pune_X_train_tensor = torch.tensor(pune_X_train, dtype=torch.float32)
pune_y_train_tensor = torch.tensor(pune_y_train, dtype=torch.float32).unsqueeze(1)
pune_X_test_tensor  = torch.tensor(pune_X_test, dtype=torch.float32)
pune_y_test_tensor  = torch.tensor(pune_y_test, dtype=torch.float32).unsqueeze(1)

# Defining the ANN Model

In [73]:
class RNNModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super(RNNModel, self).__init__()
        self.rnn = nn.RNN(input_size, hidden_size, num_layers,
                          dropout=dropout, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        out = self.fc(out[:, -1, :])  # Use the last time step
        return out

# Training the Model with the Hyperparameter Tuning done using Optuna

In [74]:
def objective(trial, X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor):
    hidden_size = trial.suggest_int("hidden_size", 32, 128)
    num_layers = trial.suggest_int("num_layers", 1, 3)
    dropout = trial.suggest_float("dropout", 0.0, 0.5) if num_layers > 1 else 0.0
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
    epochs = trial.suggest_int("epochs", 50, 150)

    model = RNNModel(input_size=X_train_tensor.shape[2], hidden_size=hidden_size,
                     num_layers=num_layers, dropout=dropout)

    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    dataset = torch.utils.data.TensorDataset(X_train_tensor, y_train_tensor)
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        preds = model(X_test_tensor)
        mse = criterion(preds, y_test_tensor).item()
    return mse

# Applying the Model for Bengaluru

In [75]:
blr_study = optuna.create_study(direction="minimize")
blr_study.optimize(lambda trial: objective(trial, blr_X_train_tensor, blr_y_train_tensor, blr_X_test_tensor, blr_y_test_tensor), n_trials=30)

[I 2025-04-21 19:30:42,555] A new study created in memory with name: no-name-dbe7a33d-ddc1-42b5-b7e4-4cd8b25bdc93
[I 2025-04-21 19:31:01,587] Trial 0 finished with value: 7.669424057006836 and parameters: {'hidden_size': 74, 'num_layers': 1, 'lr': 0.0003147103965798452, 'batch_size': 32, 'epochs': 63}. Best is trial 0 with value: 7.669424057006836.
[I 2025-04-21 19:31:31,594] Trial 1 finished with value: 24.14845848083496 and parameters: {'hidden_size': 58, 'num_layers': 2, 'dropout': 0.2910979587562912, 'lr': 0.0001070850204122122, 'batch_size': 32, 'epochs': 53}. Best is trial 0 with value: 7.669424057006836.
[I 2025-04-21 19:32:29,397] Trial 2 finished with value: 7.495942115783691 and parameters: {'hidden_size': 64, 'num_layers': 2, 'dropout': 0.3020744314457103, 'lr': 0.00014593234328751393, 'batch_size': 16, 'epochs': 62}. Best is trial 2 with value: 7.495942115783691.
[I 2025-04-21 19:34:39,271] Trial 3 finished with value: 25.99503517150879 and parameters: {'hidden_size': 110, 

In [78]:
blr_best_trial = blr_study.best_trial

In [79]:
print(f"Best MSE for Bengaluru : {blr_best_trial.value:.4f}")
print(f"Best Parameters for Bengaluru : {blr_best_trial.params}\n")

Best MSE for Bengaluru : 7.4526
Best Parameters for Bengaluru : {'hidden_size': 96, 'num_layers': 2, 'dropout': 0.2895052867270662, 'lr': 0.00024249855361833475, 'batch_size': 32, 'epochs': 116}



# Applying for Hyderabad

In [80]:
hyd_study = optuna.create_study(direction="minimize")
hyd_study.optimize(lambda trial: objective(trial, hyd_X_train_tensor, hyd_y_train_tensor, hyd_X_test_tensor, hyd_y_test_tensor), n_trials=30)

[I 2025-04-21 21:22:34,844] A new study created in memory with name: no-name-e8b60d17-4b62-4845-9d02-ea9c95743b41
[I 2025-04-21 21:23:18,927] Trial 0 finished with value: 9.564298629760742 and parameters: {'hidden_size': 76, 'num_layers': 2, 'dropout': 0.4632823052481112, 'lr': 0.0014499180891583204, 'batch_size': 64, 'epochs': 110}. Best is trial 0 with value: 9.564298629760742.
[I 2025-04-21 21:25:16,430] Trial 1 finished with value: 10.738288879394531 and parameters: {'hidden_size': 63, 'num_layers': 2, 'dropout': 0.21785240738280132, 'lr': 0.0006202718638469477, 'batch_size': 16, 'epochs': 121}. Best is trial 0 with value: 9.564298629760742.
[I 2025-04-21 21:25:28,637] Trial 2 finished with value: 19.9092960357666 and parameters: {'hidden_size': 53, 'num_layers': 1, 'lr': 0.0002709029590472435, 'batch_size': 64, 'epochs': 71}. Best is trial 0 with value: 9.564298629760742.
[I 2025-04-21 21:26:03,440] Trial 3 finished with value: 8.981602668762207 and parameters: {'hidden_size': 34,

In [81]:
hyd_best_trial = hyd_study.best_trial

In [82]:
print(f"4Best MSE for Hyderabad : {hyd_best_trial.value:.4f}")
print(f"Best Parameters for Hyderabad : {hyd_best_trial.params}\n")

4Best MSE for Hyderabad : 8.9383
Best Parameters for Hyderabad : {'hidden_size': 125, 'num_layers': 1, 'lr': 0.00014426584783540312, 'batch_size': 16, 'epochs': 99}



# Applying for Pune

In [84]:
pune_study = optuna.create_study(direction="minimize")
pune_study.optimize(lambda trial: objective(trial, pune_X_train_tensor, pune_y_train_tensor, pune_X_test_tensor, pune_y_test_tensor), n_trials=30)

[I 2025-04-21 21:56:35,116] A new study created in memory with name: no-name-7c142361-9280-4d55-8ccd-7c22b4a326e9
[I 2025-04-21 21:57:48,684] Trial 0 finished with value: 14.343682289123535 and parameters: {'hidden_size': 61, 'num_layers': 3, 'dropout': 0.4654696446038544, 'lr': 0.0013171483513862439, 'batch_size': 32, 'epochs': 85}. Best is trial 0 with value: 14.343682289123535.
[I 2025-04-21 21:59:19,008] Trial 1 finished with value: 13.088348388671875 and parameters: {'hidden_size': 103, 'num_layers': 3, 'dropout': 0.4958670176763779, 'lr': 0.00033719744675590497, 'batch_size': 64, 'epochs': 121}. Best is trial 1 with value: 13.088348388671875.
[I 2025-04-21 22:00:14,021] Trial 2 finished with value: 49.73429870605469 and parameters: {'hidden_size': 76, 'num_layers': 3, 'dropout': 0.20265638278404108, 'lr': 0.007227869029108852, 'batch_size': 64, 'epochs': 91}. Best is trial 1 with value: 13.088348388671875.
[I 2025-04-21 22:00:36,956] Trial 3 finished with value: 48.89981460571289

In [85]:
pune_best_trial = pune_study.best_trial

In [86]:
print(f"Best MSE for Pune : {pune_best_trial.value:.4f}")
print(f"Best Parameters for Pune : {pune_best_trial.params}\n")

Best MSE for Pune : 10.0943
Best Parameters for Pune : {'hidden_size': 51, 'num_layers': 1, 'lr': 0.0002433665169949093, 'batch_size': 16, 'epochs': 96}



# Evaluating the Model and Visualizing the Results

In [87]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [88]:
def evaluate_predictions(y_true, y_pred):
    y_true = y_true.squeeze()
    y_pred = y_pred.squeeze()
    
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    # NSE
    nse = 1 - np.sum((y_true - y_pred)**2) / np.sum((y_true - np.mean(y_true))**2)
    
    # RSR = RMSE / STDEV of observed
    rsr = rmse / np.std(y_true)
    
    # PBIAS
    pbias = 100 * np.sum(y_true - y_pred) / np.sum(y_true)

    return {
        "MSE": mse,
        "MAE": mae,
        "R²": r2,
        "NSE": nse,
        "RSR": rsr,
        "PBIAS": pbias
    }

In [89]:
def plot_predictions(y_true, y_pred, title="Prediction vs Ground Truth"):
    plt.figure(figsize=(10, 5))
    plt.plot(y_true.squeeze(), label="True", alpha=0.7)
    plt.plot(y_pred.squeeze(), label="Predicted", alpha=0.7)
    plt.title(title)
    plt.xlabel("Time Step")
    plt.ylabel("LST")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [90]:
from optuna.visualization.matplotlib import plot_optimization_history

def plot_optuna_loss(study, title="Optuna Loss Over Trials"):
    fig = plot_optimization_history(study)
    fig.gca().set_title(title)
    plt.tight_layout()
    plt.show()

In [105]:
def train_final_model(X_train, y_train, X_test, y_test, best_params):
    model = RNNModel(
        input_size=X_train.shape[2],
        hidden_size=best_params["hidden_size"],
        num_layers=best_params["num_layers"],
        dropout=best_params.get("dropout", 0.0)
    )

    optimizer = torch.optim.Adam(model.parameters(), lr=best_params["lr"])
    criterion = nn.MSELoss()

    dataset = torch.utils.data.TensorDataset(X_train, y_train)
    loader = torch.utils.data.DataLoader(dataset, batch_size=best_params["batch_size"], shuffle=True)

    model.train()
    for epoch in range(best_params["epochs"]):
        for X_batch, y_batch in loader:
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

    return model

In [92]:
import plotly.graph_objects as go
def plot_predictions_plotly(y_true, y_pred, title="Prediction vs Ground Truth (Test Data)"):
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        y=y_true.squeeze(),
        mode='lines',
        name='True',
        line=dict(color='blue')
    ))

    fig.add_trace(go.Scatter(
        y=y_pred.squeeze(),
        mode='lines',
        name='Predicted',
        line=dict(color='orange')
    ))

    fig.update_layout(
        title=title,
        xaxis_title="Time Step (Test Set)",
        yaxis_title="LST",
        legend=dict(x=0.01, y=0.99),
        template='plotly_white',
        height=500,
        width=1000
    )

    fig.show()

In [93]:
blr_best_params   = blr_best_trial.params
pune_best_params  = pune_best_trial.params
hyd_best_params   = hyd_best_trial.params

In [96]:
blr_model = train_final_model(
    blr_X_train_tensor, blr_y_train_tensor,
    blr_X_test_tensor, blr_y_test_tensor,
    blr_best_params
)

In [97]:
blr_model.eval()

RNNModel(
  (rnn): RNN(5, 96, num_layers=2, batch_first=True, dropout=0.2895052867270662)
  (fc): Linear(in_features=96, out_features=1, bias=True)
)

In [98]:
with torch.no_grad():
    y_pred_blr = blr_model(blr_X_test_tensor).numpy()
    y_true_blr = blr_y_test_tensor.numpy()

In [99]:
metrics_blr = evaluate_predictions(y_true_blr, y_pred_blr)
print("Bangalore Metrics:")
for k, v in metrics_blr.items():
    print(f"{k}: {v:.4f}")

Bangalore Metrics:
MSE: 7.6302
MAE: 2.1223
R²: 0.6714
NSE: 0.6714
RSR: 0.5733
PBIAS: -1.6913


In [100]:
plot_predictions_plotly(y_true_blr, y_pred_blr, title="Bangalore's Prediction vs Ground Truth (Test)")

In [106]:
hyd_model = train_final_model(
    hyd_X_train_tensor, hyd_y_train_tensor,
    hyd_X_test_tensor, hyd_y_test_tensor,
    hyd_best_params
)

In [107]:
hyd_model.eval()

RNNModel(
  (rnn): RNN(5, 125, batch_first=True)
  (fc): Linear(in_features=125, out_features=1, bias=True)
)

In [108]:
with torch.no_grad():
    y_pred_hyd = hyd_model(hyd_X_test_tensor).numpy()
    y_true_hyd = hyd_y_test_tensor.numpy()

In [109]:
metrics_hyd = evaluate_predictions(y_true_hyd, y_pred_hyd)
print("Hyderabad Metrics:")
for k, v in metrics_hyd.items():
    print(f"{k}: {v:.4f}")

Hyderabad Metrics:
MSE: 9.1709
MAE: 2.2992
R²: 0.5975
NSE: 0.5975
RSR: 0.6344
PBIAS: -1.2168


In [110]:
plot_predictions_plotly(y_true_hyd, y_pred_hyd, title="Hyderabad's Prediction vs Ground Truth (Test)")

In [111]:
pune_model = train_final_model(
    pune_X_train_tensor, pune_y_train_tensor,
    pune_X_test_tensor, pune_y_test_tensor,
    pune_best_params
)

In [112]:
pune_model.eval()

RNNModel(
  (rnn): RNN(5, 51, batch_first=True)
  (fc): Linear(in_features=51, out_features=1, bias=True)
)

In [113]:
with torch.no_grad():
    y_pred_pune = pune_model(pune_X_test_tensor).numpy()
    y_true_pune = pune_y_test_tensor.numpy()

In [114]:
metrics_pune = evaluate_predictions(y_true_pune, y_pred_pune)
print("Pune's Metrics:")
for k, v in metrics_pune.items():
    print(f"{k}: {v:.4f}")

Pune's Metrics:
MSE: 10.3293
MAE: 2.3591
R²: 0.7715
NSE: 0.7715
RSR: 0.4780
PBIAS: -1.5183


In [115]:
plot_predictions_plotly(y_true_pune, y_pred_pune, title="Pune's Prediction vs Ground Truth (Test)")

# Storing the model for future use

In [120]:
os.makedirs('../Models/RNN', exist_ok=True)

In [121]:
torch.save(blr_model.state_dict(), "../Models/RNN/blr_rnn_model.pth")

In [122]:
torch.save(hyd_model.state_dict(), "../Models/RNN/hyd_rnn_model.pth")

In [123]:
torch.save(pune_model.state_dict(), "../Models/RNN/pune_rnn_model.pth")